In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain.tools import tool
from typing import Dict,Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def recipe_search(query:str) -> Dict[str, Any]:

    """ Search the web for the Food Recipe Information"""

    return tavily_client.search(query)

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

system_prompt = """
You are the Chef who has the recipes to make food with any Ingredients that is available.
Rules:
1. Give the recipe one by one in steps
2. Include timings of the steps(example :- cook on medium heat for 5 minutes, marinate for 30 minutes)
3. Only Use the Ingredients mentioned by the user
"""

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    system_prompt=system_prompt,
    tools = [recipe_search],
    checkpointer = InMemorySaver()
)

In [ ]:
from langchain.messages import HumanMessage
from pprint import pprint


while True:
    raw_text = input("Put your query here\n")
    if raw_text.lower() == "quit":
        print("Chef : Goodbye")
        break

    question = HumanMessage(content=raw_text)
    config = {"configurable": {"thread_id": "1"}}

    response = agent.invoke(
        {"messages" : [question]},
        config
    )

    print(response['messages'][-1].content)

